In [ ]:
# Import
import os
import pandas as pd
import numpy as np

from pathlib import Path

from CellPacking.tissuegeneration import sheet_init, symetric_circular
from CellPacking.dynamics import (Compression, Compressiony, 
                                  AnisotropicLineTension, 
                                  ShearPlanarGeometry, 
                                  PlaneBarrierElasticity)

from tyssue import Sheet
from tyssue import PlanarGeometry
from tyssue.solvers import QSSolver
from tyssue.solvers.viscous import EulerSolver
from tyssue.behaviors.event_manager import EventManager
from tyssue.behaviors.sheet.basic_events import reconnect, reconnect_3D, check_tri_faces
from tyssue.dynamics import model_factory, effectors
from tyssue.core.history import HistoryHdf5 


import matplotlib.pyplot as plt
from tyssue.draw import sheet_view
from CellPacking.plot import superimpose_sheet_view
from CellPacking.plot import sheet_view as ply_sheet_view

from tyssue.generation import extrude 
from tyssue import Monolayer
from CellPacking.dynamics import ShearMonolayerGeometry
from tyssue.io.hdf5 import save_datasets

from tyssue.io.hdf5 import load_datasets
from tyssue.io.meshes import save_triangular_mesh

In [ ]:
SIM_DIR = Path('/mnt/sda1/Sophie/1-CellPacking/PRE_Revision/3D_Flat/20251009_3D_QS_Extension_double_high')

try:
    os.mkdir(SIM_DIR)
except FileExistsError:
    pass

In [ ]:
# %pdb
# pd.__version__

# Apical sheet init

In [ ]:
def tissue_init(phi, noise):
    
    apical_sheet, border = symetric_circular(10, -100, phi, 0,  noise=noise)

    apical_sheet.face_df['prefered_perimeter'] = 3 * np.sqrt(apical_sheet.face_df['prefered_area'])

    ## apical surface at the equilibrium
    # Solver
    solver_qs = QSSolver(with_t1=False, with_t3=False, with_collisions=False)

    manager = EventManager()
    manager.append(reconnect)
    # Model
    model = model_factory(
        [
            effectors.FaceAreaElasticity,
            effectors.PerimeterElasticity, 
        ])
    for i in range(50):
        manager.execute(apical_sheet)
        res = solver_qs.find_energy_min(apical_sheet, PlanarGeometry, model, periodic=False, options={"gtol": 1e-8})
        if res["success"] is False:
            print (i, res["success"])
        apical_sheet.vert_df[["x", "y"]] += np.random.normal(scale=1e-3, size=(apical_sheet.Nv, 2))
        PlanarGeometry.update_all(apical_sheet)
        manager.update()


    ## Monolayer creation
    apical_sheet.face_df['z'] = 1
    apical_sheet.edge_df['z'] = 1
    apical_sheet.vert_df['z'] = 1

    extruded = extrude(apical_sheet.datasets, method='translation', vector=[0, 0, -2])
    monolayer = Monolayer('mono', extruded)

    monolayer.sanitize(trim_borders=True, order_edges=True)
    monolayer.validate()


    monolayer.face_df['prefered_area'] = monolayer.face_df.loc[0,'prefered_area']
    monolayer.face_df['prefered_perimeter'] = 3*np.sqrt(monolayer.face_df['prefered_area'])
    monolayer.face_df['area_elasticity'] = 1
    monolayer.face_df['perimeter_elasticity'] = 0.5


    

    monolayer.vert_df['barrier_elasticity'] = 280
    monolayer.vert_df['z_barrier'] = 0.6

    monolayer.update_specs({"settings":{"dt":0.01,
                                         'threshold_length': 0.1,
                                         'p_4': 1,
                                         'p_5p': 1,
                                          "nrj_norm_factor": 1.0, 
                                       'multiplier':3, },
                          "cell": {
                                    "x": 0.0,
                                    "y": 0.0,
                                    "z": 0.0,
                                    "is_alive": True,
                                    "prefered_volume": 0.7,
                                    "volume": 0.7,
                                    "volume_elasticity": 0.5,
                                    "z_barrier":1.1,
                                    },
                          "edge": {
                                    "dx": 0.0,
                                    "srce": 0,
                                    "face": 0,
                                    "dy": 0.0,
                                    "ny": 0.0,
                                    "nx": 0.0,
                                    "length": 0.0,
                                    "nz": 0.0,
                                    "cell": 0,
                                    "sub_volume": 0.0,
                                    "dz": 0.0,
                                    "sub_area": 0.0,
                                    "trgt": 0},
                          "vert": {
                                    "x": 0.0,
                                    "is_active": True,
                                    "z": 0.0,
                                    "y": 0.0},
                          "face": {
                                    "x": 0.0,
                                    "is_alive": True,
                                    "z": 0.0,
                                    "y": 0.0,}
                        })


    ShearMonolayerGeometry.update_all(monolayer)

    monolayer.face_df['prefered_area'] = monolayer.face_df['area']
    monolayer.face_df['prefered_perimeter'] = 3*np.sqrt(monolayer.face_df['prefered_area'])

    ShearMonolayerGeometry.update_all(monolayer)

    # Manager
    manager = EventManager('face')#, track_event=False)

    monolayer.get_opposite_faces()

    monolayer.get_opposite_faces()
    edge_opp_face = monolayer.upcast_face(monolayer.face_df['opposite'])
    monolayer.edge_df['is_border']=False
    monolayer.edge_df.loc[edge_opp_face[edge_opp_face==-1].index, 'is_border']=True

    monolayer.edge_df['opposite'] = pd.to_numeric(monolayer.edge_df['opposite'])
    monolayer.edge_df['z'] = pd.to_numeric(monolayer.edge_df['z'])

    ShearMonolayerGeometry.update_all(monolayer)

    # monolayer equilibrium before apply forces 
    # Model
    model = model_factory(
        [
#                 PlaneBarrierElasticity,
    #         effectors.LineTension,
            effectors.FaceAreaElasticity,
            effectors.PerimeterElasticity,
            effectors.CellVolumeElasticity,
        ],
    )
    manager.append(reconnect_3D)
    solver_qs = QSSolver(with_t1=False, with_t3=False, with_collisions=False)

    for i in range(10):
        manager.execute(monolayer)

        res = solver_qs.find_energy_min(monolayer, ShearMonolayerGeometry, model, periodic=False, options={"gtol": 1e-8})
        if res["success"] is False:
            print (i, res["success"])
        monolayer.vert_df[["x", "y"]] += np.random.normal(scale=1e-3, size=(monolayer.Nv, 2))
        save_triangular_mesh('monolayer0.19.vtk', monolayer)
        ShearMonolayerGeometry.update_all(monolayer)
        manager.update()


    return monolayer


def simu_process(SIM_DIR, monolayer_, r, g, ratio, compress):
    
    monolayer=monolayer_.copy(deep_copy=True)
    sim_save_dir = SIM_DIR/str(r)
    try:
        os.mkdir(sim_save_dir)
    except FileExistsError:
        pass
    
    try:
        os.mkdir(sim_save_dir/str(compress))
    except FileExistsError:
        pass

    gamma_0 = g
    monolayer.cell_df["prefered_volume"] = 0.7
    monolayer.cell_df["volume_elasticity"] = 10
    monolayer.update_specs({"edge": {"gamma_0": g}})
    monolayer.edge_df['gamma_0'] = pd.to_numeric(monolayer.edge_df['gamma_0'])
    monolayer.edge_df['gamma_0'] = monolayer.edge_df['gamma_0'].replace(-100, g)
    monolayer.face_df['prefered_perimeter'] = ratio*np.sqrt(monolayer.face_df['prefered_area'])

    lat_face = monolayer.face_df[monolayer.face_df["segment"]=="lateral"].index
    monolayer.face_df.loc[lat_face, 'prefered_perimeter'] = 3.*np.sqrt(monolayer.face_df.loc[lat_face, 'prefered_area'])
    
    monolayer.specs['settings']['file_text'] = os.path.join(sim_save_dir/str(compress),'output_t1.txt')
    
    # Apply ratio -> prefered perimeter change
    model = model_factory(
        [
#                 PlaneBarrierElasticity,
#             effectors.LineTension,
            effectors.FaceAreaElasticity,
            effectors.PerimeterElasticity,
            effectors.CellVolumeElasticity,
            Compression,
            Compressiony
        ],
    )
    monolayer.vert_df["compression"] = compress

    solver_qs = QSSolver(with_t1=False, with_t3=False, with_collisions=False)
    # Manager
    manager = EventManager('face')#, track_event=False)
    manager.append(reconnect_3D)

## ADD INITIAL COMPRESSION TO THE TISSUE
    for i in range(100):
        print('------------TEMPS------------')
        print(i)
        manager.execute(monolayer)
        lat_face = monolayer.face_df[monolayer.face_df["segment"]=="lateral"].index
        monolayer.face_df.loc[lat_face, 'prefered_perimeter'] = 3.*np.sqrt(monolayer.face_df.loc[lat_face, 'prefered_area'])
        res = solver_qs.find_energy_min(monolayer, ShearMonolayerGeometry, model, periodic=False, options={"gtol": 1e-8})
        if res["success"] is False:
            print (i, res["success"])

        monolayer.vert_df[["x", "y"]] += np.random.normal(scale=1e-3, size=(monolayer.Nv, 2))
        manager.update()
        save_datasets(os.path.join(sim_save_dir/str(compress),'monolayer'+str(i)+'.hf5'), monolayer)
    
    
    ## Apply forces
    # Model

    model = model_factory(
        [
#                 PlaneBarrierElasticity,
            effectors.LineTension,
            effectors.FaceAreaElasticity,
            effectors.PerimeterElasticity,
            effectors.CellVolumeElasticity,
            Compression,
            Compressiony
        ],
    )

    solver_qs = QSSolver(with_t1=False, with_t3=False, with_collisions=False)
    # Manager
    manager = EventManager('face')#, track_event=False)
    manager.append(reconnect_3D)
    

    for i in range(200):
        i=i+100
        print('------------TEMPS------------')
        print(i)
        manager.execute(monolayer)
        lat_face = monolayer.face_df[monolayer.face_df["segment"]=="lateral"].index
        monolayer.face_df.loc[lat_face, 'prefered_perimeter'] = 3.5*np.sqrt(monolayer.face_df.loc[lat_face, 'prefered_area'])
        res = solver_qs.find_energy_min(monolayer, ShearMonolayerGeometry, model, periodic=False, options={"gtol": 1e-8})
        if res["success"] is False:
            print (i, res["success"])

        monolayer.vert_df[["x", "y"]] += np.random.normal(scale=1e-3, size=(monolayer.Nv, 2))
        manager.update()
        save_datasets(os.path.join(sim_save_dir/str(compress),'monolayer'+str(i)+'.hf5'), monolayer)


In [ ]:
from joblib import Parallel, delayed
import multiprocessing
from datetime import datetime


global_start=datetime.now()
print ("start : " + str(global_start))
num_cores = multiprocessing.cpu_count()


repeat = np.arange(0, 15) # (0,11)

gammas = 0.2
ratio = 3.
compress = [0, 0.001, 0.003, 0.005, 0.007, 0.009, 0.011, 0.013, 0.015,]
#0.017, 0.019, 0.021, 0.023, 0.025, 0.027, 0.029, 
#          0.031, 0.033, 0.035, 0.037, 0.039, 0.041, 0.043, 0.045, 0.047, 0.049, 0.051]
compress = [0, 0.005, 0.015, 0.025, 0.051, 0.01]
compress = [0, 0.015, 0.021, 0.025, 0.031, 0.035]


# Number of cells in x and y axis
nx = 40
ny = 40
noise = 0.3
phi = np.pi/2

for r in repeat:
    monolayer = tissue_init(phi, noise)
    print("------------------------------------")
    print("init tissue")
    results = Parallel(n_jobs=12)(delayed(simu_process)(
        SIM_DIR, monolayer, r, gammas, ratio, round(-c, 4)) for c in compress)

# simu_process(repeat[0][0], gammas[0][0], phi, noise)
global_end = datetime.now()
print ("end : " + str(global_end))
print ('Duree totale d execution : \n\t\t')
print (global_end-global_start)

In [ ]:
import time
from plyer import notification
notification.notify(
    title = "ALERT!!!",
    message = "It is finish",
    timeout=10
)

In [ ]:
#  raise SystemExit("Stop right there!")


In [ ]:
sim_save_dir = SIM_DIR/str(5)  
g=-0.031

monolayer_d = load_datasets(os.path.join(sim_save_dir/str(g),'monolayer199.hf5'))
monolayer = Monolayer("mono", monolayer_d)

fig = superimpose_sheet_view(monolayer.get_sub_sheet("apical"), monolayer.get_sub_sheet("basal"))



import plotly.express as px
import plotly.graph_objects as go

# 1233, 1237, 1239, 1241, 1246, 1249, 1252, 1255, 1260, 1266, 1271
# vert_id=1233
# print(len(monolayer.edge_df[(monolayer.edge_df["srce"]==vert_id) | (monolayer.edge_df["trgt"]==vert_id)]))
# fig.add_traces(go.Scatter(x=monolayer.vert_df.loc[vert_id][list("x")].to_numpy(),
#                          y=monolayer.vert_df.loc[vert_id][list("y")].to_numpy(),
#                          mode='markers', 
#                          marker=dict(size=20)
#                          )
#               )

fig.show()

In [ ]:
from CellPacking.plot import sheet_view as view3d
from tyssue.io.meshes import save_mesh

save_mesh("mono_"+str(g)+".ply", monolayer)



In [ ]:
ls


# Analyse

In [ ]:

result = pd.DataFrame(columns = ['repeat', 'ratio_p', 'nb_change', 'tot_cell'])

repeat = np.arange(8) # (0,11)

gammas = 0.2
ratio = 3.
# compress = [0, 0.005, 0.015, 0.025]
compress = [0, 0.015, 0.021, 0.025, 0.031, 0.035]
# compress = [0, 0.005, 0.015, 0.025, 0.051, 0.01]

# 0.017, 0.019, 0.021, 0.023, 0.025, 0.027, 0.029, 
          # 0.031, 0.033, 0.035, 0.037, 0.039, 0.041, 0.043, 0.045, 0.047, 0.049, 0.051]


for r in repeat:
    sim_save_dir = SIM_DIR/str(r)   
    for ratio in compress:
        g=-round(ratio,4)
        dir_ = sim_save_dir/str(g)
        
        
        monolayer_d = load_datasets(os.path.join(sim_save_dir/str(g),'monolayer1.hf5'))
        monolayer = Monolayer("mono", monolayer_d)
        count0 = len(np.unique(monolayer.edge_df[(monolayer.edge_df['face'].isin(monolayer.face_df[monolayer.face_df['num_sides']==3].index)) &
                  (monolayer.edge_df['segment']=='lateral') & (monolayer.edge_df['face'].isin(monolayer.face_df[monolayer.face_df['area']>0.01].index))]['cell']))
        
        
        try : 
            monolayer_d = load_datasets(os.path.join(sim_save_dir/str(g),'monolayer299.hf5'))
        except: 
            monolayer_d = load_datasets(os.path.join(sim_save_dir/str(g),'monolayer150.hf5'))
        monolayer = Monolayer("mono", monolayer_d)
        count = len(np.unique(monolayer.edge_df[(monolayer.edge_df['face'].isin(monolayer.face_df[monolayer.face_df['num_sides']==3].index)) &
                  (monolayer.edge_df['segment']=='lateral') & (monolayer.edge_df['face'].isin(monolayer.face_df[monolayer.face_df['area']>0.01].index))]['cell']))
        
        count = count-count0
        result = pd.concat([result, pd.DataFrame({'repeat':r,
                                                  'compression':g, 
                                                  'nb_change':count,
                                                 'tot_cell': monolayer.Nc},
                          index=[0])],
                          ignore_index=True)
        
        save_triangular_mesh('monolayer'+str(g)+'.vtk', monolayer)
        
result['pourcentage'] = result['nb_change']/result['tot_cell']*100
result.to_csv(os.path.join(SIM_DIR, 'result_pourcentage_min.csv'))

In [ ]:
result = pd.read_csv(os.path.join(SIM_DIR, 'result_pourcentage_min.csv'))


In [ ]:
result

In [ ]:
result['pourcentage2'] = result['pourcentage']

In [ ]:
fig, ax = plt.subplots()

ax.scatter(result['compression'], result['pourcentage'],c=result.repeat, cmap="rainbow", alpha=0.8)
ax.set_xlabel('compression')
ax.set_ylabel('% of cell that have neighbouring change')

ax.plot(result.groupby("compression").mean().index, 
        result.groupby("compression").mean()["pourcentage"].to_numpy(), 
        c='black')


ax.bar(result.groupby("compression").mean().index, 
       result.groupby("compression").mean().pourcentage, 
       yerr = result.groupby("compression").std().pourcentage, 
       align='center',
       alpha=0,
       ecolor='black',
       capsize=5)

# _=ax.set_yticks([0, 10,20,30, 40])
# _=ax.set_xticks([2,2.5,3.,3.5,4])
ax.set_xlim(-0.001, 0.016)
# ax.set_ylim(-0.5, 45)
# fig.set_size_inches((10,10))
# ax.set_aspect('equal')
fig.savefig(os.path.join(SIM_DIR,"lateral_fixe_3d_perimeter_t1_wth_basal.eps"), dpi=150)
fig.savefig(os.path.join(SIM_DIR,"lateral_fixe_3d_perimeter_t1_wth_basal.png"), dpi=150)

In [ ]:
fig, ax = plt.subplots()

ax.scatter(result['compression'], result['pourcentage2'],c=result.repeat, cmap="rainbow", alpha=0.8)
ax.set_xlabel('compression')
ax.set_ylabel('% of cell that have neighbouring change')

ax.plot(result.groupby("compression").mean().index, 
        result.groupby("compression").mean()["pourcentage2"].to_numpy(), 
        c='black')


ax.bar(result.groupby("compression").mean().index, 
       result.groupby("compression").mean().pourcentage2, 
       yerr = result.groupby("compression").std().pourcentage2, 
       align='center',
       alpha=0,
       ecolor='black',
       capsize=5)

# _=ax.set_yticks([0, 10,20,30, 40])
# _=ax.set_xticks([2,2.5,3.,3.5,4])
ax.set_xlim(0.001, -0.016)
# ax.set_ylim(-0.5, 45)
# fig.set_size_inches((10,10))
# ax.set_aspect('equal')
# fig.savefig("3d_perimeter_t1.eps", dpi=150)

In [ ]:
result['pourc_change'] = (result['nb_change']/2)/result['tot_cell']*100
# result.drop("group", axis=1, inplace=True)
fig, ax = plt.subplots()
ax.scatter(result.ratio_p,
           result.pourc_change, 
           c=result.repeat, cmap="rainbow", alpha=0.8)

ax.plot(result[["pourc_change", "compression"]].groupby("compression").mean().index, 
        result[["pourc_change", "compression"]].groupby("compression").mean()["pourc_change"].to_numpy(), 
        c='black')


ax.bar(result.groupby("compression").mean().index, 
       result.groupby("compression").mean().pourc_change, 
       yerr = result.groupby("compression").std().pourc_change, 
       align='center',
       alpha=0,
       ecolor='black',
       capsize=5)

ax.set_xlabel('shape index')
ax.set_ylabel('%')

# _=ax.set_yticks([0, 10,20,30])
# _=ax.set_xticks([0.05,0.1,0.15,0.2])
ax.set_xlim(-0.0, -0.035)
# ax.set_ylim(-0.01, 30)
fig.set_size_inches((10,10))
# ax.set_aspect('equal')
# fig.savefig("2d_gamma_t1.eps", dpi=150)

In [ ]:
fig, ax = plt.subplots()
# ax.plot(result['gamma'], result['pourcentage'], '.', markersize=10, color='black')
ax.set_xlabel('compression')
ax.set_ylabel('% of cell that have neighbouring change')

ax.plot(result.groupby('compression').mean().index, result.groupby('compression').mean()['pourcentage'], 
        '.',color='red',  markersize=10, label='mean')


ax.errorbar(result.groupby('compression').mean().index,
            result.groupby('compression').mean()['pourcentage'],
            result.groupby('compression').std()['pourcentage'],
            linestyle='None', fmt='-o', color='red')
fig.set_size_inches((10,10))

# fig.savefig(SIM_DIR/'result.eps', dpi=300)

In [ ]:
result.to_csv(os.path.join(SIM_DIR, 'result_count.csv'))

# Time plot

In [ ]:
nb_changes_tot = []
angle_tot = []
r = 0
g = 0.0


repeat = np.arange(2,4) # (0,11)

gammas = 0.2
ratio = 3.
compress = [0, 0.001, 0.003, 0.005, 0.007, 0.009, 0.011, 0.013, 0.015,]
# 0.017, 0.019, 0.021, 0.023, 0.025, 0.027, 0.029, 
          # 0.031, 0.033, 0.035, 0.037, 0.039, 0.041, 0.043, 0.045, 0.047, 0.049, 0.051]


for r in repeat:
    sim_save_dir = SIM_DIR/str(r)   
    for ratio in compress:
        
        g=-round(ratio,4)
        dir_ = sim_save_dir/str(g)

        nb_changes = []
        for i in range(299):
            if os.path.isfile(os.path.join(sim_save_dir/str(g),'monolayer'+str(i)+'.hf5')):
                monolayer_d = load_datasets(os.path.join(sim_save_dir/str(g),'monolayer'+str(i)+'.hf5'))

                monolayer = Monolayer("mono", monolayer_d)
                count = len(np.unique(monolayer.edge_df[(monolayer.edge_df['face'].isin(monolayer.face_df[monolayer.face_df['num_sides']==3].index)) &
                              (monolayer.edge_df['segment']=='lateral') & (monolayer.edge_df['face'].isin(monolayer.face_df[monolayer.face_df['area']>0.01].index))]['cell']))
                nb_changes.append(count/monolayer.Nc*100)

        nb_changes_tot.append(nb_changes)

        tri_face_ids = np.unique(monolayer.edge_df[(monolayer.edge_df['face'].isin(monolayer.face_df[monolayer.face_df['num_sides']==3].index)) &
                          (monolayer.edge_df['segment']=='lateral') & (monolayer.edge_df['face'].isin(monolayer.face_df[monolayer.face_df['area']>0.01].index))]['cell'])

        angle0=[]
        for f in tri_face_ids :
            dx = monolayer.edge_df[(monolayer.edge_df['face']==f) & 
                            (((monolayer.edge_df['sz']>0.4) & (monolayer.edge_df['tz']>0.4)) |
                             ((monolayer.edge_df['sz']<-0.4) & (monolayer.edge_df['tz']<-0.4)))]['dx'].to_numpy()
            dy = monolayer.edge_df[(monolayer.edge_df['face']==f) & 
                            (((monolayer.edge_df['sz']>0.4) & (monolayer.edge_df['tz']>0.4)) |
                             ((monolayer.edge_df['sz']<-0.4) & (monolayer.edge_df['tz']<-0.4)))]['dy'].to_numpy()
            if len(dx)!=0:
                angle0.append(np.arctan2(dy, dx)[0]*180/np.pi)        

        angle_tot.append(angle0)


In [ ]:
len(nb_changes_tot)

In [ ]:
fig, ax = plt.subplots()
i=0
for nb_tot in nb_changes_tot[:18]:
    ax.plot(np.array(nb_tot), label=str(np.round(i,2)))
    i+=0.02
# ax.plot(nb_changes2, label='0.2')
ax.legend()
ax.set_xlabel('\'time\'')
ax.set_ylabel('% of cell ')

# fig.set_size_inches((10,10))

In [ ]:
fig, ax = plt.subplots()
i=2
for nb_tot in nb_changes_tot[:18]:
    ax.plot(np.array(nb_tot[:100]), label=str(np.round(i,1)))
    i+=0.2
# ax.plot(nb_changes2, label='0.2')
ax.legend()
ax.set_xlabel('\'time\'')
ax.set_ylabel('% of cell ')

# fig.set_size_inches((10,10))

In [ ]:
fig, ax = plt.subplots()
i=2
for nb_tot in nb_changes_tot[:18]:
    ax.plot(np.array(nb_tot[100:200]), label=str(np.round(i,1)))
    i+=0.2
# ax.plot(nb_changes2, label='0.2')
ax.legend()
ax.set_xlabel('\'time\'')
ax.set_ylabel('AB-T1%')

# fig.set_size_inches((10,10))
fig.savefig("t1_time_with_forces.png",dpi=300)
fig.savefig("t1_time_with_forces.eps",dpi=300)

In [ ]:
plt.hist(monolayer.face_df[monolayer.face_df["segment"]=="apical"]["prefered_perimeter"]/np.sqrt(monolayer.face_df[monolayer.face_df["segment"]=="apical"]["prefered_area"]), alpha=0.5)
plt.hist(monolayer.face_df[monolayer.face_df["segment"]=="basal"]["prefered_perimeter"]/np.sqrt(monolayer.face_df[monolayer.face_df["segment"]=="basal"]["prefered_area"]), alpha=0.5)
plt.hist(monolayer.face_df[monolayer.face_df["segment"]=="lateral"]["prefered_perimeter"]/np.sqrt(monolayer.face_df[monolayer.face_df["segment"]=="lateral"]["prefered_area"]), alpha=0.5)

In [ ]:
len(angle_tot)

In [ ]:
import numpy as np

def calculate_angle_between_segments(point_a, point_b, point_c, point_d):
    # Define vectors
    vector_ab = np.array([point_b[i] - point_a[i] for i in range(3)])
    vector_cd = np.array([point_d[i] - point_c[i] for i in range(3)])
    
    # Calculate dot product
    dot_product = np.dot(vector_ab, vector_cd)
    
    # Calculate magnitudes
    magnitude_ab = np.linalg.norm(vector_ab)
    magnitude_cd = np.linalg.norm(vector_cd)
    
    # Ensure no division by zero
    if magnitude_ab == 0 or magnitude_cd == 0:
        raise ValueError("One of the segments is of zero length.")
    
    # Calculate cosine of the angle
    cos_theta = dot_product / (magnitude_ab * magnitude_cd)
    
    # Ensure the value is within the valid range for arccos due to numerical errors
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    
    # Calculate the angle in radians and convert to degrees
    theta = np.arccos(cos_theta)
    angle_degrees = np.degrees(theta)
    
    return angle_degrees


In [ ]:
sim_save_dir = SIM_DIR/str(5)   
monolayer_d = load_datasets(os.path.join(sim_save_dir/str(0),'monolayer299.hf5'))
mono = Monolayer("mono", monolayer_d)


angles_2 = []
lengths_2 = []
remove_i = []

pos_A = []
pos_B = []
for i in mono.edge_df[(mono.edge_df["segment"]=="lateral") & (mono.edge_df["is_border"]==False)].index:
    
    if (mono.edge_df.loc[i][["srce", "trgt"]].min(),
        mono.edge_df.loc[i][["srce", "trgt"]].max()) not in remove_i:
    
        z_dist = np.abs(mono.vert_df.loc[mono.edge_df.loc[i]["trgt"]]["z"] - mono.vert_df.loc[mono.edge_df.loc[i]["srce"]]["z"])
        if (mono.edge_df.loc[i]["length"]>0.2) and z_dist>0.5:

            A = mono.vert_df.loc[mono.edge_df.loc[i]["trgt"]][list("xyz")]
            B = mono.vert_df.loc[mono.edge_df.loc[i]["srce"]][list("xyz")]
            pos_A.append(A)
            pos_B.append(B)

            remove_i.append((mono.edge_df.loc[i][["srce", "trgt"]].min(),
                             mono.edge_df.loc[i][["srce", "trgt"]].max()))

            C = (0, 0, 0)
            D = (0, 0, 1)
            lengths_2.append(np.linalg.norm(np.array([B[i] - A[i] for i in range(3)])))

            angle = calculate_angle_between_segments(A, B, C, D)
            angles_2.append(angle)
            
            
            
monolayer_d = load_datasets(os.path.join(sim_save_dir/str(0.013),'monolayer299.hf5'))
mono = Monolayer("mono", monolayer_d)


angles_4 = []
lengths_4 = []
remove_i = []

pos_A = []
pos_B = []
z_dists = []
for i in mono.edge_df[(mono.edge_df["segment"]=="lateral") & (mono.edge_df["is_border"]==False)].index:
    
    if (mono.edge_df.loc[i][["srce", "trgt"]].min(),
        mono.edge_df.loc[i][["srce", "trgt"]].max()) not in remove_i:
    
        z_dist = np.abs(mono.vert_df.loc[mono.edge_df.loc[i]["trgt"]]["z"] - mono.vert_df.loc[mono.edge_df.loc[i]["srce"]]["z"])
        z_dists.append(z_dist)
        if (mono.edge_df.loc[i]["length"]>0.2) and z_dist>0.5:
        
            A = mono.vert_df.loc[mono.edge_df.loc[i]["trgt"]][list("xyz")]
            B = mono.vert_df.loc[mono.edge_df.loc[i]["srce"]][list("xyz")]
            pos_A.append(A)
            pos_B.append(B)

            remove_i.append((mono.edge_df.loc[i][["srce", "trgt"]].min(),
                             mono.edge_df.loc[i][["srce", "trgt"]].max()))

            C = (0, 0, 0)
            D = (0, 0, 1)
            lengths_4.append(np.linalg.norm(np.array([B[i] - A[i] for i in range(3)])))

            angle = calculate_angle_between_segments(A, B, C, D)
            angles_4.append(angle)

In [ ]:
fig, ax = plt.subplots()
# angles_2 = angle_tot[0]
aas_2 = [t+180 if t<0 else t for t in angles_2]
aas_2 = [180-t if t>90 else t for t in aas_2]
# _=ax.hist(angles_2, bins = 50, alpha=0.5)
_=ax.hist(aas_2, bins = 50, alpha=0.5)

# angles_4 = angle_tot[10]
aas_4 = [t+180 if t<0 else t for t in angles_4]
aas_4 = [180-t if t>90 else t for t in aas_4]
_=ax.hist(aas_4, bins = 50, alpha=0.5)

In [ ]:
import seaborn as sns

df = pd.DataFrame([np.concatenate([aas_2, aas_4]),
                  np.concatenate([np.repeat(2, len(aas_2)), 
                                  np.repeat(4, len(aas_4))])]).T
df.columns=["angle", "cond"]

sns.set_style('whitegrid')
fig, ax = plt.subplots(figsize=(4, 6))
sns.violinplot(x="cond", y="angle", data = df,
               color='#DDDDDD', inner="quart"
              )


# sns.stripplot(x='cond', y='angle', data=df, 
#               jitter=True, linewidth=1, )


medians = df.groupby('cond').median().reset_index()
q25 = df.groupby('cond').quantile(0.25).reset_index()
q75 = df.groupby('cond').quantile(0.75).reset_index()

sns.swarmplot(x='cond', y='angle', data=medians, 
              color='white', edgecolor='black', linewidth=1, size=8)

sns.swarmplot(x='cond', y='angle', data=q25, 
              color='red', edgecolor='black', linewidth=1, size=6)

sns.swarmplot(x='cond', y='angle', data=q75, 
              color='red', edgecolor='black', linewidth=1, size=6)

from scipy.stats import ttest_ind
print(ttest_ind(aas_2, aas_4))
# fig.savefig(os.path.join(SIM_DIR, "violin_plot_orientation_lateral_edges.png"))
# fig.savefig(os.path.join(SIM_DIR, "violin_plot_orientation_lateral_edges.eps"))
# fig.savefig("violin_plot_orientation_lateral_edges.eps")

# Result V3 - tenseur 

In [ ]:
np.arange(12)

In [ ]:
from scipy.linalg import eig, inv

result = pd.DataFrame(columns = ['repeat', 'ratio_p', 'T_i', 'eigen'])

repeat = np.arange(8)
ratios = [0, 0.015, 0.021, 0.025, 0.031, 0.035]
# ratios = [0, 0.005, 0.015, 0.025, 0.051, 0.01]


for r in repeat:
    print(r)
    sim_save_dir = SIM_DIR/str(r)   
    for ratio in ratios:
        print(ratio)
        g=-round(ratio,4)
        # g=round(ratio, 1)
        dir_ = sim_save_dir/str(g)
        try : 
            monolayer_d = load_datasets(os.path.join(sim_save_dir/str(g),'monolayer299.hf5'))
        except: 
            monolayer_d = load_datasets(os.path.join(sim_save_dir/str(g),'monolayer150.hf5'))
        monolayer = Monolayer("mono", monolayer_d)
        id_new_edges = monolayer.edge_df[(monolayer.edge_df['face'].isin(monolayer.face_df[monolayer.face_df['num_sides']==3].index)) 
                                                 & (monolayer.edge_df['face'].isin(monolayer.face_df[monolayer.face_df['area']>0.01].index))
                                                 &   (((monolayer.edge_df['sz']>0.4) & (monolayer.edge_df['tz']>0.4)) |
                                                     ((monolayer.edge_df['sz']<-0.4) & (monolayer.edge_df['tz']<-0.4))) 
                                                 ].index

        # Pour chaque edge apical (ou basal) d'une face triangulaire
        mm_apical = []
        mm_basal = []
        for id_ in id_new_edges:
            # Recuperation des faces voisines
            id_face_neighbours = monolayer.get_neighbors(monolayer.edge_df.loc[id_]['face'], elem='face')
            neighbouring_face = monolayer.face_df.loc[list(id_face_neighbours)]

            # recuperation des faces uniquement apicale(ou basale)
            faces = []
            segment = ""
            for nf in neighbouring_face.index : 
                if ((monolayer.edge_df[monolayer.edge_df['face']==nf]['sz'] >  0.2).all() & (monolayer.edge_df[monolayer.edge_df['face']==nf]['tz'] >  0.2).all()):
                    if monolayer.face_df.loc[nf]['opposite']==-1:
                        faces.append(nf)
                        segment = "apical"
                elif ((monolayer.edge_df[monolayer.edge_df['face']==nf]['sz'] < -0.2).all() & (monolayer.edge_df[monolayer.edge_df['face']==nf]['tz'] < -0.2).all()):
                    if monolayer.face_df.loc[nf]['opposite']==-1:
                        faces.append(nf)
                        segment = 'basal'

            if len(faces)==2:
                # Creation de la matrice m
                X, Y, Z = monolayer.face_df.loc[faces[0]][list("xyz")] - monolayer.face_df.loc[faces[1]][list("xyz")]


                m = np.array([[X**2,  X*Y , X*Z ],
                              [Y*X  , Y**2, Y*Z ],
                              [Z*X  , Z*Y , Z**2]
                            ])
                m = m[:2,:2]
                if segment == 'apical':
                    mm_apical.append(m)
                elif segment == 'basal':
                    mm_basal.append(m)
                    
        mm_apical = np.array(mm_apical)
        mm_basal  = np.array(mm_basal)
        T_i = mm_apical.shape[0] * mm_apical.mean(axis=0) - mm_basal.shape[0] * mm_basal.mean(axis=0)
        
        
        #Eigen value calculation
        # Diagonalisation
        if np.isnan(T_i).all():
            diag_m = np.zeros((2, 2))
            for i in range(0,len(vals)):
                diag_m[i,i] = np.nan
        else:
            vals, vecs = eig(T_i)
            diag_m = np.zeros((2, 2))
            for i in range(0,len(vals)):
                diag_m[i,i] = vals[i].real
       
        
        result = pd.concat([result, pd.DataFrame({'repeat':r,
                                                  'ratio_p':g, 
                                                  'T_i':[T_i],
                                                  'eigen':[diag_m]})],
                          ignore_index=True)

In [ ]:
result


In [ ]:
# Plot 
eigen_x = []
eigen_y = []
for i in range(result.shape[0]):
    eigen_x.append(np.min((result.iloc[i]['eigen'][0,0], result.iloc[i]['eigen'][1,1])))
    eigen_y.append(np.max((result.iloc[i]['eigen'][0,0], result.iloc[i]['eigen'][1,1])))
#     eigen_x.append(result.iloc[i]['eigen'][0,0])
#     eigen_y.append(result.iloc[i]['eigen'][1,1])

fig, ax = plt.subplots()
fig.set_size_inches((10,10))
i=0
for g in ratios:
    ax.scatter(eigen_x[i::len(ratios)], eigen_y[i::len(ratios)], s=50, label=round(g, 3))
    i+=1
ax.set_xlabel('eigne_x')
ax.set_ylabel('eigen_y')
ax.legend()
# fig.savefig(os.path.join(SIM_DIR,"eigens.eps"), dpi=150)
# fig.savefig(os.path.join(SIM_DIR,"eigens.png"), dpi=150)

In [ ]:
# Matrice T sommée sur tous les réplicats
result_sum = result.groupby("ratio_p").sum()[['T_i']]/10
result_sum['eigen'] = np.repeat("", result_sum.shape[0])

# Recalcul des valeurs propres sur la somme des T_i (=> Lambda_X(<T_i>))
for id_, val in result_sum.iterrows():
    if type(val['T_i']) == np.ndarray :
        if np.isnan(val['T_i']).any():
            result_sum.loc[id_]['eigen'] = np.nan
        else:
            vals, vecs = eig(val['T_i'])
            diag_m = np.zeros((2,2))
            for i in range(0,len(vals)):
                diag_m[i,i] = vals[i].real
            result_sum.loc[id_]['eigen'] = diag_m
    else:
        result_sum.loc[id_]['eigen'] = np.nan

    

# plot
fig, ax = plt.subplots()
fig.set_size_inches((10,10))
for id_, val in result_sum.iterrows():
    if np.isnan(val["eigen"]).any() == False:
        ax.scatter(id_, np.min((val['eigen'][0,0], val['eigen'][1,1])), color='black', s=50)

ax.set_xlabel('ratio_p')
ax.set_ylabel('eigen_x')


# fig.savefig(SIM_DIR/'gamma_eigenX.png', dpi=150)

In [ ]:
result_sum

In [ ]:
# Bootstrap pour calculer l'intervalle de confiance
eigen_x = []
eigen_y = []
for i in range(result.shape[0]):
    eigen_x.append(np.min((result.iloc[i]['eigen'][0,0], result.iloc[i]['eigen'][1,1])))

from scipy.stats import bootstrap

result_bootstrap = pd.DataFrame(columns= ['ratio_p', 'ci_l', 'ci_h', 'std_error'])
i=0
for g in ratios:
    res = bootstrap((eigen_x[i::11],), 
                    np.std, 
                    confidence_level=0.95, 
                    random_state=1,
                    method="percentile", 
                   )
    
    result_bootstrap = pd.concat([result_bootstrap, 
                                  pd.DataFrame.from_dict({'ratio_p':round(g,2), 
                                                'ci_l':res.confidence_interval[0],
                                                'ci_h':res.confidence_interval[1],
                                                'std_error':res.standard_error,
                                               }, orient='index').T],
                          ignore_index=True)
    i+=1

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches((10,10))
result_bootstrap.plot('ratio_p', ['ci_l', 'ci_h', 'std_error'], ax=ax)
# fig.savefig(SIM_DIR/'result_bootstrap.png', dpi=150)

In [ ]:
fig, ax = plt.subplots()
i=0
nb_val = len(ratios)
for r in repeat:
    # st = i*9
    st = i*nb_val
    ax.scatter(ratios, eigen_x[st:st+nb_val:],s=50)
    i+=1
ax.set_ylabel('eigen_x')
ax.set_xlabel('gamma')
fig.set_size_inches((5,5))

In [ ]:
len(eigen_x[i::nb_val]), len(np.repeat(g,nb_val))

In [ ]:
fig, ax = plt.subplots()
i=0
for g in repeat:
    ax.scatter(np.repeat(g,nb_val), eigen_x[i::np.max(repeat)+1], s=50, color='black')
    i+=1
ax.set_ylabel('eigen_x')
ax.set_xlabel('G')
fig.set_size_inches((7,7))
# fig.savefig(SIM_DIR/'gamma_eigenX_replicats.png', dpi=150)


# fig, ax = plt.subplots()
# i=0
# for g in gammas:
#     ax.plot(np.repeat(g,10), eigen_y[i::21], '.', color='black')
#     i+=1
# ax.set_xlabel('gamma')
# ax.set_ylabel('eigen_y')

In [ ]:
eigen_x = []
eigen_y = []
for i in range(result.shape[0]):
    eigen_x.append(np.min((result.iloc[i]['eigen'][0,0], result.iloc[i]['eigen'][1,1])))
    eigen_y.append(np.max((result.iloc[i]['eigen'][0,0], result.iloc[i]['eigen'][1,1])))
    
result['Lambda_x'] = eigen_x
result['Lambda_y'] = eigen_y
result

In [ ]:

eigen_x = []
eigen_y = []
for i in range(result_sum.shape[0]):
    if np.isnan(result_sum.iloc[i]['eigen']).any():
        eigen_x.append(np.nan)
        eigen_y.append(np.nan)
    else:
        eigen_x.append(np.min((result_sum.iloc[i]['eigen'][0,0], result_sum.iloc[i]['eigen'][1,1])))
        eigen_y.append(np.max((result_sum.iloc[i]['eigen'][0,0], result_sum.iloc[i]['eigen'][1,1])))
    
result_sum['Lambda_x'] = eigen_x
result_sum['Lambda_y'] = eigen_y
result_sum

In [ ]:
result.to_csv(SIM_DIR/"result_ti.csv")
result_sum.to_csv(SIM_DIR/"result_t_sum.csv")
result_bootstrap.to_csv(SIM_DIR/"result_bootstrap.csv")

In [ ]:
paths = ["/mnt/sda1/Sophie/1-CellPacking/PRE_Revision/3D_Flat/20251009_3D_QS_Extension_double_high/",]
        # "/mnt/sda1/Sophie/1-CellPacking/0-Simulations-to-sort/20250320_3D_QS_without_HI_PerimeterImpactVolume_lateralfixe_3/",
        # "/mnt/sda1/Sophie/1-CellPacking/0-Simulations-to-sort/20250320_3D_QS_without_HI_PerimeterImpactVolume_lateralfixe_3_5/", 
        # "/mnt/sda1/Sophie/1-CellPacking/0-Simulations-to-sort/20250320_3D_QS_without_HI_PerimeterImpactVolume_lateralfixe_4/"]

shi = [2.5, 3, 3.5, 4]

res_dataframe = []

i=0
for sdir in paths : 
    result = pd.read_csv(os.path.join(sdir, 'result_count.csv'))
    result_sum = pd.read_csv(os.path.join(sdir, "result_ti.csv"))
    result_final = result.copy(deep=True)
    result_final['P'] = ((result_sum['Lambda_y']-result_sum['Lambda_x'])/result_final['tot_cell']*100).to_numpy()
    result_final.drop(['Unnamed: 0.1', 'Unnamed: 0',"pourcentage2", "pourc_change"], axis=1, inplace=True	)
    result_final["shape_index_lateral"] = shi[i]

    if i ==0 : 
        res_dataframe = result_final.copy(deep=True)
    else:
        res_dataframe = pd.concat([res_dataframe, result_final])
    i+=1
res_dataframe.reset_index(inplace=True, drop=True)
res_dataframe.to_csv("/mnt/sda1/Sophie/1-CellPacking/PRE_Revision/3D_Flat/20251009_3D_QS_Extension_double_high/result_all.csv")

In [ ]:
res_dataframe

In [ ]:
#result from first analysis
result = pd.read_csv(os.path.join(SIM_DIR, 'result_count.csv'))
result_sum = pd.read_csv(SIM_DIR/"result_ti.csv")
result_final = result.copy(deep=True)
result_final['P'] = ((result_sum['Lambda_y']-result_sum['Lambda_x'])/result_final['tot_cell']*100).to_numpy()
result_final.groupby("compression").mean()

In [ ]:
result_sum.shape

In [ ]:
fig, ax = plt.subplots()
ax.scatter(result_final['pourcentage'], result_final['P'],s=50, color='black')
ax.scatter(result_final.groupby("compression").mean()['pourcentage'], result_final.groupby("compression").mean()['P'],s=50, color='red')
ax.set_xlabel("AB-T1(%)")
ax.set_ylabel("γ")

fig.savefig(os.path.join(SIM_DIR, "gamma_vs_abt1.png"))
fig.savefig(os.path.join(SIM_DIR, "gamma_vs_abt1.eps"))

In [ ]:
result_sum

In [ ]:
group_pourcent = result_final.groupby('compression').mean()['pourcentage']
result_final['trace'] = (result_sum['Lambda_y']+result_sum['Lambda_x']).to_numpy()
result_final

In [ ]:
fig, axes = plt.subplots(nrows=2, sharex=True)
ax = axes[0]
threshold = 0.012
ax.scatter(result_final[result_final["compression"]<threshold]['pourcentage'], 
           result_final[result_final["compression"]<threshold]['P'],s=50, c=result_final[result_final["compression"]<threshold]['compression'], cmap="RdPu")


ax.set_ylabel("γ")

ax = axes[1]
im = ax.scatter(result_final[result_final["compression"]<threshold]['pourcentage'], 
                result_final[result_final["compression"]<threshold]['trace'],s=50,c=result_final[result_final["compression"]<threshold]['compression'], cmap="RdPu")
# ax.scatter(group_pourcent, 
#            result_final.groupby('ratio_p').mean()['trace'],s=50, color='red')
ax.set_xlabel("AB-T1(%)")
ax.set_ylabel("Q")

fig.colorbar(im, ax= axes,  orientation='vertical', label='compression', fraction=.1)
fig.savefig(os.path.join(SIM_DIR, "tensor_vs_abt1.png"))
fig.savefig(os.path.join(SIM_DIR, "tensor_vs_abt1.eps"))

In [ ]:
fig, axes = plt.subplots(nrows=2, sharex=True)
ax = axes[0]
ax.scatter(result_final['pourcentage'], 
           result_final['P'],s=50, c=-result_final['compression'], cmap="RdPu")


ax.set_ylabel("γ")

ax = axes[1]
im = ax.scatter(result_final['pourcentage'], result_final['trace'],s=50,c=-result_final['compression'], cmap="RdPu")
# ax.scatter(group_pourcent, 
#            result_final.groupby('ratio_p').mean()['trace'],s=50, color='red')
ax.set_xlabel("AB-T1(%)")
ax.set_ylabel("Q")

fig.colorbar(im, ax= axes,  orientation='vertical', label='Extension', fraction=.1)
fig.savefig(os.path.join(SIM_DIR, "tensor_vs_abt1.png"))
fig.savefig(os.path.join(SIM_DIR, "tensor_vs_abt1.eps"))

In [ ]:
fig, ax = plt.subplots()
ax.scatter(result_final['compression'], result_final['P'],s=50, color='black')
ax.scatter(result_final.groupby("compression").mean().index, result_final.groupby("compression").mean()['P'],s=50, color='red')
ax.set_xlabel("Compression")
ax.set_ylabel("γ")
fig.savefig(os.path.join(SIM_DIR, "gamma_shapeindex.png"))
fig.savefig(os.path.join(SIM_DIR, "gamma_shapeindex.eps"))

In [ ]:
fig, ax = plt.subplots()

import random
get_colors = lambda n: ["#%06x" % random.randint(0, 0xFFFFFF) for _ in range(n)]
color_list = get_colors(100)

cc = []
for rp in (result_final["compression"]*10).to_numpy():
    cc.append(color_list[int(rp)])


ax.scatter(result_final['compression'], result_final['P'], s=50, color=cc)
ax.plot(result_final.groupby("compression").mean().index, result_final.groupby("compression").mean()['P'], 
        linewidth=4, color='black')

yerr = [result_final.groupby("compression").std()["P"]/2,
        result_final.groupby("compression").std()["P"]/2] 

ax.errorbar(result_final.groupby("compression").mean().index, 
            result_final.groupby("compression").mean()['P'],  
            yerr=yerr, capsize=3, fmt="k.", ecolor = "black")

ax.set_xlabel("Shape index")
ax.set_ylabel("γ")
ax.grid(False)
fig.savefig(os.path.join(SIM_DIR, "gamma_shapeindex2.png"))
fig.savefig(os.path.join(SIM_DIR, "gamma_shapeindex2.eps"))

In [ ]:
result_final.to_csv(os.path.join(SIM_DIR,"result_final_extension.csv"))
result_final

## 3D view

In [ ]:
SIM_DIR = Path('/mnt/sda1/Sophie/1-CellPacking/PRE_Revision/3D_Flat/20250730_3D_QS_Compression/')
sim_save_dir = SIM_DIR
try:
    os.mkdir(sim_save_dir)
except FileExistsError:
    pass

In [ ]:
for t in range (300):
    monolayer_d = load_datasets(os.path.join(SIM_DIR/str(0)/str(0.011),'monolayer'+str(t)+'.hf5'))
    monolayer = Monolayer("mono", monolayer_d)
    mono = Monolayer("mono", monolayer_d)
    
    save_mesh("0.011_vtk/mono_"+str(t)+".ply", mono)

In [ ]:
from CellPacking.plot import sheet_view as view3d
from tyssue.io.meshes import save_mesh

monolayer_d = load_datasets(os.path.join(sim_save_dir/str(1)/str(2.0),'monolayer299.hf5'))
monolayer = Monolayer("mono", monolayer_d)
mono = Monolayer("mono", monolayer_d)

save_mesh("mono_2.0.ply", mono)

In [ ]:
sheet = monolayer.get_sub_sheet('apical')
save_mesh("mono_2.0_apical.ply", sheet)

sheet = monolayer.get_sub_sheet('basal')
save_mesh("mono_2.0_basal.ply", sheet)


In [ ]:
def segment_index(mono, column, segment, element):
    df = getattr(mono, "{}_df".format(element))
    return df[df[column] == segment].index

def get_sub_mono(mono, segment):
    """Returns a :class:`Sheet` object of the corresponding
    segment

    Parameters
    ----------
    segment: str, the corresponding segment, wether 'apical' or 'basal'

    """
    datasets = {
        element: mono.datasets[element].loc[segment_index(mono, "half_cut", segment, element)]
        for element in ["cell", "edge", "face", "vert"]
    }
    specs = {k: mono.specs[k] for k in ["face", "edge", "vert", "settings"]}
    return Monolayer(mono.identifier + str(segment), datasets, specs)



monolayer.cell_df["half_cut"] = 0
monolayer.edge_df["half_cut"] = 0
monolayer.face_df["half_cut"] = 0
monolayer.vert_df["half_cut"] = 0

half_cell = monolayer.cell_df[monolayer.cell_df['y']>0].index
monolayer.cell_df.loc[half_cell, "half_cut"] = 1
monolayer.edge_df.loc[monolayer.edge_df[monolayer.edge_df["cell"].isin(half_cell)].index, "half_cut"] = 1
monolayer.face_df.loc[monolayer.edge_df[monolayer.edge_df["cell"].isin(half_cell)]["face"], "half_cut"] = 1
monolayer.vert_df.loc[monolayer.edge_df[monolayer.edge_df["cell"].isin(half_cell)]["srce"], "half_cut"] = 1
monolayer.vert_df.loc[monolayer.edge_df[monolayer.edge_df["cell"].isin(half_cell)]["trgt"], "half_cut"] = 1

save_mesh("mono_2.0_half.ply", get_sub_mono(monolayer, 1))


In [ ]:
import ipyvolume as ipv
from tyssue import config
from tyssue.draw.ipv_draw import sheet_view as sheet_view_3d

color='white'
ipv.clear()
draw_spec = config.draw.sheet_spec()
draw_spec['face']['visible'] = True
draw_spec['face']['color'] = color

fig, meshes = sheet_view_3d(mono, **draw_spec)
fig = ipv.gcf()

fig.anglex = 1.0
fig.angley = 0.2
fig.anglez = 0.1

ipv.show()

In [ ]:
from tyssue.io.meshes import save_mesh


#extract one cell
for cell_id in mono.cell_df.index:
# cell_id = 12
    datasets = {}

    datasets["cell"] = mono.cell_df[mono.cell_df["id"]==cell_id].copy()
    datasets["edge"] = mono.edge_df[mono.edge_df["cell"]==cell_id].copy()
    datasets["face"] = mono.face_df.loc[np.unique(mono.edge_df[mono.edge_df["cell"]==cell_id]["face"])].copy()
    datasets["vert"] = mono.vert_df.loc[mono.edge_df[mono.edge_df["cell"]==cell_id]["srce"].unique()].copy()

    subsheet = Monolayer("subsheet", datasets,)
    subsheet.reset_index()
    subsheet.reset_topo()


    save_mesh("2.0/"+str(cell_id)+".ply", subsheet)